# Exploratory Data Analysis (EDA) - Fake Review Detection

This notebook walks through the analysis of the review dataset. It investigates class balance, review length differences between genuine and deceptive reviews, and visualizes the most common terms found in each class.

### Objectives:
1. **Load and inspect** the review dataset.
2. **Examine class distribution** (Real vs. Fake reviews).
3. **Analyze text statistics** (character count, word count, sentence count).
4. **Clean the text** using the preprocessing pipeline.
5. **Identify common terms** in each category (N-grams analysis).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Append path to import utils
sys.path.append(os.path.abspath('..'))
from utils.preprocess import clean_text

# Set plotting styles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Load and Inspect Dataset

We will locate and load the CSV dataset from the `dataset` directory.

In [ ]:
from IPython import display
dataset_path = '../dataset/fake_reviews.csv'

if not os.path.exists(dataset_path):
    # If not found under default name, search for any CSV file in dataset directory
    import glob
    csv_files = glob.glob('../dataset/*.csv')
    if csv_files:
        dataset_path = csv_files[0]
        print(f"Found dataset at: {dataset_path}")
    else:
        print("No CSV dataset found. Please place your CSV file under the dataset/ directory!")

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset shape: {df.shape}")
    display(df.head())
else:
    df = None

## 2. Check and Standardize Columns

Let's identify the text and label columns, handle null values, and map labels to a standard binary representation:
- **`0`**: Genuine / Real / Truthful reviews
- **`1`**: Fake / Computer-Generated / Deceptive reviews

In [ ]:
if df is not None:
    # Attempt auto-detection
    from train_model import detect_columns, preprocess_labels
    
    text_col, label_col = detect_columns(df)
    print(f"Auto-detected Text Column: '{text_col}', Label Column: '{label_col}'")
    
    # Drop missing values in crucial columns
    df = df.dropna(subset=[text_col, label_col]).copy()
    
    # Map labels
    df['target'] = preprocess_labels(df, label_col)
    df = df.dropna(subset=['target'])
    df['target'] = df['target'].astype(int)
    
    print(f"Processed dataset shape: {df.shape}")

## 3. Class Balance Analysis

Let's visualize the balance between real and fake reviews.

In [ ]:
if df is not None:
    class_counts = df['target'].value_counts()
    print("Class Counts:")
    print(class_counts)
    
    plt.figure(figsize=(6, 5))
    sns.countplot(x='target', data=df, palette='viridis')
    plt.title('Distribution of Genuine (0) vs. Fake (1) Reviews')
    plt.xticks([0, 1], ['Genuine (0)', 'Fake (1)'])
    plt.xlabel('Class Label')
    plt.ylabel('Number of Reviews')
    plt.tight_layout()
    plt.show()

## 4. Text Length Analysis

It is common for fake and real reviews to have different length characteristics. Real reviews are often more detailed and descriptive, or conversely, fake reviews written by bots might have standard word counts. Let's calculate length features:
- **`char_count`**: Total characters in the review.
- **`word_count`**: Total words in the review.

In [ ]:
if df is not None:
    df['char_count'] = df[text_col].astype(str).str.len()
    df['word_count'] = df[text_col].astype(str).apply(lambda x: len(x.split()))
    
    print("Length Statistics for Genuine Reviews:")
    display(df[df['target'] == 0][['char_count', 'word_count']].describe())
    
    print("\nLength Statistics for Fake Reviews:")
    display(df[df['target'] == 1][['char_count', 'word_count']].describe())

Let's plot the distribution of word count across the two classes.

In [ ]:
if df is not None:
    plt.figure(figsize=(12, 5))
    
    # Histograms
    sns.histplot(data=df, x='word_count', hue='target', kde=True, bins=50, multiple='dodge', palette='muted')
    plt.title('Word Count Distribution by Class')
    plt.xlabel('Word Count')
    plt.ylabel('Count')
    plt.legend(labels=['Fake (1)', 'Genuine (0)'])
    plt.xlim(0, df['word_count'].quantile(0.98)) # Filter extreme outliers for visualization
    plt.tight_layout()
    plt.show()

## 5. Preprocessing Verification

Let's see how our text preprocessing module `clean_text` behaves on a sample review. It lowers characters, removes punctuation/numbers, removes stopwords, and performs lemmatization.

In [ ]:
if df is not None and not df.empty:
    sample_review = df[text_col].iloc[0]
    cleaned_sample = clean_text(sample_review)
    
    print("--- ORIGINAL ---")
    print(sample_review)
    print("\n--- PREPROCESSED & LEMMATIZED ---")
    print(cleaned_sample)

## 6. Common Words Analysis (N-grams)

Let's extract and examine the most common unigrams (single words) and bigrams (two-word phrases) in genuine and deceptive reviews using `scikit-learn`'s `CountVectorizer`.

In [ ]:
if df is not None:
    from sklearn.feature_extraction.text import CountVectorizer
    
    # Preprocess all texts in the dataset
    print("Preprocessing all reviews for N-gram analysis...")
    df['cleaned'] = df[text_col].apply(clean_text)
    
    def plot_top_ngrams(texts, title, n=2, top_k=15):
        vec = CountVectorizer(ngram_range=(n, n)).fit(texts)
        bag_of_words = vec.transform(texts)
        sum_words = bag_of_words.sum(axis=0)
        words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
        words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)[:top_k]
        
        ngram_df = pd.DataFrame(words_freq, columns=['N-gram', 'Frequency'])
        
        sns.barplot(x='Frequency', y='N-gram', data=ngram_df, palette='plasma')
        plt.title(title)
        plt.xlabel('Frequency')
        plt.ylabel('Phrase')
        plt.tight_layout()
        plt.show()
    
    # Top Bigrams for Genuine Reviews
    print("Plotting top bigrams for Genuine Reviews...")
    plt.figure(figsize=(10, 6))
    plot_top_ngrams(df[df['target'] == 0]['cleaned'], "Top 15 Bigrams in Genuine Reviews", n=2)
    
    # Top Bigrams for Fake Reviews
    print("Plotting top bigrams for Fake Reviews...")
    plt.figure(figsize=(10, 6))
    plot_top_ngrams(df[df['target'] == 1]['cleaned'], "Top 15 Bigrams in Fake Reviews", n=2)

## 7. Conclusions & Modeling Strategy

Based on the analysis, we can summarize:
1. **Class distribution** status (balanced or imbalanced) helps us choose suitable metrics (Accuracy vs. F1-Score) and set class weights.
2. **Length differences** can indicate stylistic characteristics. (e.g. bots might generate shorter reviews, or highly exaggerated, repetitive phrases).
3. **Common phrases** give us hints about semantic patterns. We can use a TF-IDF vectorizer to convert these patterns into numerical vectors for our Logistic Regression classifier.